<a href="https://colab.research.google.com/github/Coolguy4123/Data-Visualization-Project-3/blob/main/CS4990_Vehicle_Visualization%20(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Initialization

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("syedanwarafridi/vehicle-sales-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'vehicle-sales-data' dataset.
Path to dataset files: /kaggle/input/vehicle-sales-data


In [2]:
import pandas as pd
df = pd.read_csv(path + "/car_prices.csv")
df.head()

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


# Data Preprocessing

In [3]:
print(df.info())

print(f"\n\nShape: {df.shape}")
print(f"Columns: {df.columns}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558837 entries, 0 to 558836
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          558837 non-null  int64  
 1   make          548536 non-null  object 
 2   model         548438 non-null  object 
 3   trim          548186 non-null  object 
 4   body          545642 non-null  object 
 5   transmission  493485 non-null  object 
 6   vin           558833 non-null  object 
 7   state         558837 non-null  object 
 8   condition     547017 non-null  float64
 9   odometer      558743 non-null  float64
 10  color         558088 non-null  object 
 11  interior      558088 non-null  object 
 12  seller        558837 non-null  object 
 13  mmr           558799 non-null  float64
 14  sellingprice  558825 non-null  float64
 15  saledate      558825 non-null  object 
dtypes: float64(4), int64(1), object(11)
memory usage: 68.2+ MB
None


Shape: (558837, 16)
Column

In [4]:
print(f"Number of duplicates: {df.duplicated().sum()}")
print(f"\n\nNumber of missing values: {df.isnull().sum()}")

Number of duplicates: 0


Number of missing values: year                0
make            10301
model           10399
trim            10651
body            13195
transmission    65352
vin                 4
state               0
condition       11820
odometer           94
color             749
interior          749
seller              0
mmr                38
sellingprice       12
saledate           12
dtype: int64


In [5]:
# --- Data cleaning steps ---
# ---------------------------

# 1. Drop rows with missing values in important columns
df = df.dropna(subset=['sellingprice', 'mmr', 'odometer', 'make', 'body'])


# 2. Fill missing values in less critical columns
df['transmission'] = df['transmission'].fillna('unknown')
df['condition'] = df['condition'].fillna(df['condition'].median())
df['color'] = df['color'].fillna('unknown')
df['interior'] = df['interior'].fillna('unknown')


# 3. Drop columns that are not useful for visualization
df = df.drop(columns=['vin', 'seller', 'trim', 'model', 'saledate'])


# 4. Remove extreme outliers from key numeric columns
df = df[df['sellingprice'] < df['sellingprice'].quantile(0.99)]
df = df[df['odometer'] < df['odometer'].quantile(0.99)]
df = df.copy()


# 5. Reduce category complexity for cleaner visualizations
top_makes = df['make'].value_counts().nlargest(10).index
df['make'] = df['make'].where(df['make'].isin(top_makes), 'Other')


top_body = df['body'].value_counts().nlargest(8).index
df['body'] = df['body'].where(df['body'].isin(top_body), 'Other')


# 6. Columns for visualization
df = df[['year', 'make', 'body', 'transmission', 'condition',
         'odometer', 'color', 'interior', 'state', 'mmr',
         'sellingprice']]

print("\nNew Shape:", df.shape)
print("\nMissing values after preprocessing:\n", df.isnull().sum())


New Shape: (534662, 11)

Missing values after preprocessing:
 year            0
make            0
body            0
transmission    0
condition       0
odometer        0
color           0
interior        0
state           0
mmr             0
sellingprice    0
dtype: int64


In [6]:
# --- Encoding and Scaling ---
# ----------------------------
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
df = df.copy()

# --- 1. Encode categorical variables ---
categorical_cols = ['make', 'body', 'transmission', 'color', 'interior', 'state']

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# --- 2. Scale numerical variables ---
numerical_cols = ['year', 'condition', 'odometer', 'mmr', 'sellingprice']

scaler = MinMaxScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print(df.describe())
df.head()

                year           make           body   transmission  \
count  534662.000000  534662.000000  534662.000000  534662.000000   
mean        0.807481       5.948897       4.364090       0.260935   
std         0.152592       3.174192       1.684915       0.650555   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.720000       4.000000       3.000000       0.000000   
50%         0.880000       7.000000       5.000000       0.000000   
75%         0.920000       9.000000       5.000000       0.000000   
max         1.000000      10.000000       8.000000       2.000000   

           condition       odometer          color       interior  \
count  534662.000000  534662.000000  534662.000000  534662.000000   
mean        0.624099       0.292867       9.632112       4.010549   
std         0.274380       0.214237       6.943719       4.327312   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.479167       0.1261

,year,make,body,transmission,condition,odometer,color,interior,state,mmr,sellingprice
0,1.00,7,4,0,0.083333,0.074157,18,1,3,0.186179,0.478479
1,1.00,7,4,0,0.083333,0.041861,18,0,3,0.188907,0.478479
2,0.96,0,5,0,0.916667,0.005928,7,1,3,0.289839,0.667653
3,1.00,9,5,0,0.833333,0.063652,18,1,3,0.249830,0.617578
5,1.00,8,5,0,0.000000,0.024750,7,1,3,0.139350,0.242567


# First Visualization

In [26]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()


# ensure these columns are numeric
numeric_cols = ["odometer", "mmr", "sellingprice"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# drop rows missing the required numeric values
df_viz = df.dropna(subset=numeric_cols)

# sample the data so the plot isn't insanely messy
df_sample = df_viz.sample(200, random_state=42)
df_sample[numeric_cols] = scaler.fit_transform(df_sample[numeric_cols])
# convert make → numbers (parallel coords needs numeric colors)
df_sample["make_code"] = df_sample["make"].astype("category").cat.codes

# build the parallel-coordinates plot
fig = px.parallel_coordinates(
    df_sample,
    dimensions=["odometer", "mmr", "sellingprice"],
    color="sellingprice",
    labels={
        "odometer": "Normalized Odometer (miles)",
        "mmr": "Normalized MMR Value",
        "sellingprice": "Normalized Selling Price",
        "make_code": "Make (encoded)"
    },
    color_continuous_scale=px.colors.sequential.Plasma
)
fig.update_layout(
    margin=dict(t=100)  # increase top space
)

fig.update_layout(
    title={
        'text': "Relationship Between Mileage, Market Value, and Selling Price of Cars",
        'y': 0.98,   # moves title slightly higher
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size':20}
    },
    width=900,
    height=500
)

fig.show()

# Second Visualization

In [28]:
# average selling price by make x body type

import pandas as pd
import plotly.graph_objects as go

# decode categorical back from label-encoded integers
df_heat = df.copy()
for col in ['make', 'body']:
    df_heat[col] = label_encoders[col].inverse_transform(
        df_heat[col].astype(int)
    )


# normalize body type casing to avoid duplicate categories (e.g. 'sedan' vs 'Sedan')
df_heat['body'] = df_heat['body'].str.title()


# check the scaled range of sellingprice before rescaling
sp_min = df_heat['sellingprice'].min()
sp_max = df_heat['sellingprice'].max()
print(f"sellingprice scaled range: {sp_min:.4f} – {sp_max:.4f}")


# MinMaxScaler maps [min, max] → [0, 1], so reverse: original = scaled * (max - min) + min
# PRICE_MAX set to 99th percentile cap applied during preprocessing
PRICE_MIN = 1.0
PRICE_MAX = 75000.0
df_heat['sellingprice_dollars'] = (
    df_heat['sellingprice'] * (PRICE_MAX - PRICE_MIN) + PRICE_MIN
)


# aggregate mean selling price for each make × body type combination
pivot_price = (
    df_heat.groupby(['make', 'body'])['sellingprice_dollars']
    .mean()
    .unstack() # reshape to matrix: rows = make, cols = body type
    .round(0)
)

# build heatmap: rows = make, cols = body type, color = avg selling price
fig1 = go.Figure(data=go.Heatmap(
    z=pivot_price.values,
    x=pivot_price.columns.tolist(),
    y=pivot_price.index.tolist(),
    colorscale='Viridis',
    colorbar=dict(title='Avg Selling Price ($)'),
    hoverongaps=False,
    hovertemplate=(
        'Make: %{y}<br>'
        'Body: %{x}<br>'
        'Avg Selling Price: $%{z:,.0f}<extra></extra>'
    )
))

# set title, subtitle annotation, and axis labels
fig1.update_layout(
    title=dict(
        text='Average Selling Price by Car Make and Body Type',
        x=0.5, xanchor='center',
        font=dict(size=20)
    ),
    annotations=[dict(
        text=('Each cell shows the average selling price for a '
              'make-body combination. Darker = lower price, '
              'brighter = higher price.'),
        x=0, y=1.08, xref='paper', yref='paper',
        showarrow=False, font=dict(size=13)
    )],
    xaxis_title='Body Type',
    yaxis_title='Make',
    margin=dict(t=120),
    width=1000,
    height=500
)

# format cell text: show dollar amount or blank for missing/zero combos
import numpy as np
fig1.update_traces(text=[['' if (v == 0 or np.isnan(v)) else f'${v:,.0f}' for v in row] for row in pivot_price.values])
fig1.show()

sellingprice scaled range: 0.0000 – 1.0000


In [10]:
# average condition score by make x body type

import numpy as np

# reload condition from original df since it was overwritten during sellingprice rescaling
df_heat['condition'] = df['condition'].values

# check the scaled range of condition before rescaling
print(df_heat['condition'].min(), df_heat['condition'].max())

# MinMaxScaler maps [min, max] → [0, 1], so reverse: original = scaled * (max - min) + min
# condition original range is 1 (Poor) to 5 (Excellent)
COND_MIN = 1.0
COND_MAX = 5.0
df_heat['condition'] = df_heat['condition'] * (COND_MAX - COND_MIN) + COND_MIN

# aggregate mean condition score for each make × body type combination
pivot_condition = (
    df_heat.groupby(['make', 'body'])['condition']
    .mean()
    .unstack() # reshape to matrix: rows = make, cols = body type
    .round(2)
)

# build heatmap: rows = make, cols = body type, color = avg condition score
fig2 = go.Figure(data=go.Heatmap(
    z=pivot_condition.values,
    x=pivot_condition.columns.tolist(),
    y=pivot_condition.index.tolist(),
    colorscale='RdYlGn', # red = poor condition, green = excellent condition
    colorbar=dict(title='Avg Condition (1–5)'),
    zmin=1, zmax=5, # fix color scale to full 1–5 range
    hoverongaps=False,
    hovertemplate=(
        'Make: %{y}<br>'
        'Body: %{x}<br>'
        'Avg Condition: %{z:.2f}<extra></extra>'
    )
))

# set title, subtitle annotation, and axis labels
fig2.update_layout(
    title=dict(
        text='Average Vehicle Condition Score by Make and Body Type',
        x=0.5, xanchor='center',
        font=dict(size=20)
    ),
    annotations=[dict(
        text=('Each cell shows the average condition score (1=Poor, '
              '5=Excellent) for a make-body combination. '
              'Red = poor condition, Green = excellent condition.'),
        x=0, y=1.08, xref='paper', yref='paper',
        showarrow=False, font=dict(size=13)
    )],
    xaxis_title='Body Type',
    yaxis_title='Make',
    margin=dict(t=120)
)

# format cell text: show condition score or blank for missing combos
fig2.update_traces(text=[['' if (np.isnan(v)) else f'{v:.2f}' for v in row] for row in pivot_condition.values])
fig2.show()

0.0 0.9999999999999999


## Questions


*   Looking at the SUV selling price heatmap, which car make has the highest average selling price for SUVs?

*   Among all body types, which body type has the highest average selling price for Ford vehicles?

*   Compare the average selling price of BMW Hatchbacks vs. Ford SUVs. Which is higher and by approximately how much?

*   Looking at the condition heatmap, does the make with the highest average condition score for Sedans also have one of the highest average selling points for Sedans in the price heatmap? Describe what you observe.



